# **Configuration de Neo4j et Génération (Local Sudo-Free)**

In [ ]:
%%bash
# Installation de l'environnement (Neo4j, APOC config v5)
NEO4J_VERSION="5.18.0"
if [ ! -d "neo4j_local" ]; then
    wget -q -nc https://neo4j.com/artifact.php?name=neo4j-community-$NEO4J_VERSION-unix.tar.gz -O neo4j.tar.gz
    tar -xzf neo4j.tar.gz
    mv neo4j-community-$NEO4J_VERSION neo4j_local
    rm neo4j.tar.gz
fi
if [ ! -f "neo4j_local/plugins/apoc-$NEO4J_VERSION-core.jar" ]; then
    wget -q -nc https://github.com/neo4j/apoc/releases/download/$NEO4J_VERSION/apoc-$NEO4J_VERSION-core.jar -P neo4j_local/plugins/
fi
CONF_FILE="neo4j_local/conf/neo4j.conf"
APOC_CONF="neo4j_local/conf/apoc.conf"
if ! grep -q "dbms.security.procedures.unrestricted=apoc.\*" "$CONF_FILE"; then
    echo "dbms.security.procedures.unrestricted=apoc.*" >> "$CONF_FILE"
    echo "apoc.export.file.enabled=true" > "$APOC_CONF"
    ./neo4j_local/bin/neo4j-admin dbms set-initial-password "password"
fi
echo "[+] Environnement Neo4j prêt !"

In [ ]:
# On importe notre fichier utilitaire python local !
import adsim_utils
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import os

### Lancement de la génération des Graphes

In [ ]:
NUM_GRAPHS = 2

for i in range(1, NUM_GRAPHS + 1):
    adsim_utils.run_pipeline(i)

print("\n[+] GÉNÉRATION TERMINÉE ! Toutes les matrices sont prêtes.")

### Affichage

In [ ]:
jsonl_file = "./Dataset/graph_1.json"
if os.path.exists(jsonl_file):
    nodes, edges = adsim_utils.load_jsonl(jsonl_file)
    G = nx.DiGraph()
    color_map = []
    
    for n in nodes:
        labels = n.get('labels', [])
        G.add_node(n['id'], labels=labels)
        if 'Computer' in labels: color_map.append('lightblue')
        elif 'User' in labels: color_map.append('lightgreen')
        elif 'Group' in labels: color_map.append('orange')
        elif 'Domain' in labels: color_map.append('purple')
        else: color_map.append('gray')
            
    for e in edges:
        start_id = e['start']['id'] if isinstance(e.get('start'), dict) else e['start']
        end_id = e['end']['id'] if isinstance(e.get('end'), dict) else e['end']
        G.add_edge(start_id, end_id, label=e.get('label', e.get('type', '')))
        
    plt.figure(figsize=(16, 12))
    pos = nx.spring_layout(G, k=0.3, iterations=50, seed=42)
    nx.draw_networkx_nodes(G, pos, node_size=60, node_color=color_map, edgecolors='black')
    nx.draw_networkx_edges(G, pos, alpha=0.2, arrows=True, arrowsize=8, edge_color='gray')
    
    legend_elements = [
        mpatches.Patch(color='lightblue', label='Computers'),
        mpatches.Patch(color='lightgreen', label='Users'),
        mpatches.Patch(color='orange', label='Groups'),
        mpatches.Patch(color='purple', label='Domain')
    ]
    plt.legend(handles=legend_elements, loc='upper left')
    plt.axis('off')
    plt.show()

In [ ]:
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

instance_id = 1
structured_file = f"./Dataset/graph_{instance_id}_structured.json"
sp_file = f"./Dataset/graph_shortest_path{instance_id}.json"

if os.path.exists(structured_file):
    print(f"[*] Chargement des données structurées de l'instance {instance_id}...")
    
    with open(structured_file, 'r') as f:
        data = json.load(f)
        
    metadata = data.get('metadata', {})
    nodes_dict = data['node_registry'] 
    edge_index = data['subgraph_topology']['edge_index']
    
    # =========================================================================
    # 1. PARSING DES CHEMINS D'ATTAQUE (Avec extraction des Labels)
    # =========================================================================
    paths_by_source = {}
    sp_edge_labels_global = {}  
    
    if os.path.exists(sp_file):
        with open(sp_file, 'r') as f:
            for line in f:
                if not line.strip(): continue
                row = json.loads(line)
                
                if "p" in row and isinstance(row["p"], dict):
                    rels = row["p"].get("rels", [])
                    if not rels: continue
                        
                    first_start = rels[0].get('start', {})
                    source_id = str(first_start.get('id', first_start) if isinstance(first_start, dict) else first_start)
                    
                    if source_id not in paths_by_source:
                        paths_by_source[source_id] = set()
                        
                    for r in rels:
                        start_data = r.get('start', {})
                        end_data = r.get('end', {})
                        
                        start_id = str(start_data.get('id', start_data) if isinstance(start_data, dict) else start_data)
                        end_id = str(end_data.get('id', end_data) if isinstance(end_data, dict) else end_data)
                        
                        rel_label = r.get('label', r.get('type', 'Unknown'))
                        
                        if start_id and end_id:
                            paths_by_source[source_id].add((start_id, end_id))
                            sp_edge_labels_global[(start_id, end_id)] = rel_label
                            
    # =========================================================================
    # 2. CONSTRUCTION DU GRAPHE GLOBAL
    # =========================================================================
    G_sub = nx.DiGraph()
    idx_to_orig_id = {}
    allocations = []
    
    for idx_str, node_data in nodes_dict.items():
        idx = int(idx_str) 
        orig_id = str(node_data['original_id'])
        idx_to_orig_id[idx] = orig_id
        
        is_source = node_data.get('is_source', False)
        is_terminal = node_data.get('is_terminal', False)
        alloc_val = node_data.get('best_allocation_weight', 0.0)
        
        allocations.append(alloc_val)
        G_sub.add_node(orig_id, alloc=alloc_val, is_source=is_source, is_terminal=is_terminal)
        
    for u_idx, v_idx in edge_index:
        u_id = idx_to_orig_id[u_idx]
        v_id = idx_to_orig_id[v_idx]
        label = sp_edge_labels_global.get((u_id, v_id), "")
        G_sub.add_edge(u_id, v_id, label=label)
        
    print("[*] Calcul du Layout spatial global (peut prendre quelques secondes)...")
    pos = nx.spring_layout(G_sub, k=0.5, iterations=100, seed=42) 
    
    max_alloc = max(allocations) if allocations and max(allocations) > 0 else 1.0
    cmap = plt.cm.Reds
    norm = mcolors.Normalize(vmin=0, vmax=max_alloc)

    # =========================================================================
    # 3. AFFICHAGE (1 PLOT PAR SOURCE)
    # =========================================================================
    if not paths_by_source:
        print("[-] Aucun Shortest Path trouvé.")
    else:
        plot_number = 1
        total_plots = len(paths_by_source)
        MAX_PLOTS = 10 # Sécurité pour ne pas faire crasher le notebook
        
        for source_id, sp_edges in paths_by_source.items():
            if plot_number > MAX_PLOTS:
                print(f"[*] Limite de {MAX_PLOTS} graphiques atteinte pour éviter de surcharger l'affichage.")
                break
                
            fig, ax = plt.subplots(figsize=(15, 12))
            
            # Identifier les noeuds impliqués dans l'attaque
            sp_nodes = set([u for u, v in sp_edges] + [v for u, v in sp_edges])
            
            # --- Préparation des calques visuels (Background vs Foreground) ---
            bg_nodelist, bg_colors, bg_edgecolors, bg_sizes, bg_widths = [], [], [], [], []
            fg_nodelist, fg_colors, fg_edgecolors, fg_sizes, fg_widths = [], [], [], [], []
            
            for n in G_sub.nodes():
                alloc = G_sub.nodes[n]['alloc']
                color = cmap(norm(alloc))
                
                # C'est un noeud "important" (Source, Cible ou sur le Chemin)
                if n == source_id or G_sub.nodes[n]['is_terminal'] or n in sp_nodes:
                    fg_nodelist.append(n)
                    fg_colors.append(color)
                    if n == source_id:
                        fg_edgecolors.append('red'); fg_widths.append(4.0); fg_sizes.append(450)
                    elif G_sub.nodes[n]['is_terminal']:
                        fg_edgecolors.append('gold'); fg_widths.append(4.0); fg_sizes.append(450)
                    else:
                        fg_edgecolors.append('blue'); fg_widths.append(2.0); fg_sizes.append(250 + (alloc / max_alloc) * 200)
                # C'est un noeud "secondaire" (Décor)
                else:
                    bg_nodelist.append(n)
                    bg_colors.append(color)
                    bg_edgecolors.append('gray'); bg_widths.append(1.0); bg_sizes.append(80 + (alloc / max_alloc) * 100)
                    
            # 1. Dessiner le Background (Transparent)
            if bg_nodelist:
                nx.draw_networkx_nodes(G_sub, pos, nodelist=bg_nodelist, node_size=bg_sizes, 
                                       node_color=bg_colors, edgecolors=bg_edgecolors, linewidths=bg_widths, alpha=0.2, ax=ax)
            
            # 2. Dessiner le Foreground (Solide)
            if fg_nodelist:
                nx.draw_networkx_nodes(G_sub, pos, nodelist=fg_nodelist, node_size=fg_sizes, 
                                       node_color=fg_colors, edgecolors=fg_edgecolors, linewidths=fg_widths, alpha=1.0, ax=ax)
            
            # 3. Trier et dessiner les Arêtes
            bg_edges = [(u, v) for u, v in G_sub.edges() if (u, v) not in sp_edges]
            fg_edges = [(u, v) for u, v in G_sub.edges() if (u, v) in sp_edges]
            
            nx.draw_networkx_edges(G_sub, pos, edgelist=bg_edges, alpha=0.1, arrows=True, edge_color='gray', ax=ax)
            nx.draw_networkx_edges(G_sub, pos, edgelist=fg_edges, alpha=1.0, arrows=True, arrowsize=18, edge_color='blue', width=2.5, ax=ax)
            
            # 4. Ajouter les Labels textuels sur les arêtes d'attaque
            if fg_edges:
                edge_labels = {(u, v): G_sub.edges[u, v].get('label', '') for u, v in fg_edges}
                nx.draw_networkx_edge_labels(G_sub, pos, edge_labels=edge_labels, font_color='darkblue', font_size=9, font_weight='bold', ax=ax)
            else:
                ax.text(0.5, 0.05, "⚠️ Le chemin de cette attaque a été filtré par le sous-graphe ML.", 
                        horizontalalignment='center', transform=ax.transAxes, fontsize=12, color='red', fontweight='bold')
            
            # 5. Titre et Légendes
            budget = metadata.get('budget_limit', 'N/A')
            plt.title(f"Attack Path Isolation ({plot_number}/{total_plots})\nSource Active : {source_id} | Budget Global : {budget}", fontsize=14, fontweight='bold')
            
            legend_elements = [
                mpatches.Patch(facecolor='white', edgecolor='red', linewidth=3, label='Active Source Node'),
                mpatches.Patch(facecolor='white', edgecolor='gold', linewidth=3, label='Terminals (Domain Admins)'),
                plt.Line2D([0], [0], color='blue', lw=2.5, label='Attack Vector'),
                plt.Line2D([0], [0], color='gray', lw=1.0, alpha=0.5, label='Topology Background')
            ]
            ax.legend(handles=legend_elements, loc='upper left', framealpha=0.9)
            
            # 6. Ajouter la Colorbar pour l'allocation budgétaire
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
            cbar.set_label('Defense Budget Allocation Weight', rotation=270, labelpad=20, fontweight='bold')
            
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            plot_number += 1

else:
    print(f"[-] Le fichier {structured_file} n'existe pas.")